## Classification Model

In [21]:
import torch.nn as nn
import torch 

class MicrogliaFeatureEmbedder(nn.Module):
# get predictions using CNN 

    def __init__(self, dim=256):
        super().__init__()
        # images are b&w
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(p=0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(p=0.2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.embedded = nn.Linear(128, dim)


    def forward(self, x):
        f = self.features(x)
        f = self.gap(f).flatten(1)
        return self.embedded(f).squeeze(0)

class GatedAttention(nn.Module):
# from ilse et al
    def __init__(self, dim=256, h_dim=128):
        super().__init__()
        self.V = nn.Linear(dim, h_dim)
        self.U = nn.Linear(dim, h_dim)
        self.w = nn.Linear(h_dim,1)

    def forward(self, x):
        gated_val = torch.tanh(self.V(x)) * torch.sigmoid(self.U(x))
        vals = self.w(gated_val)

        weights = torch.softmax(vals, dim=0)
        return (weights * x).sum(dim=0, keepdim=True), weights.squeeze()

class MILClassifier(nn.Module):

    def __init__(self, dim=256):
        super().__init__()
        self.feature_embedding = MicrogliaFeatureEmbedder(dim)
        self.gated_attention = GatedAttention(dim)
        self.out_classifier = nn.Linear(dim,1)

    def forward(self, embeddings):
        z, attention = self.gated_attention(embeddings)
        pred = self.out_classifier(z)
        return pred, attention

In [22]:
from torch.utils.data import IterableDataset, Dataset
from PIL import Image
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import gc
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pyvips
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *

class CellDataset(Dataset):

    def __init__(self, imgs, parquets, embedder, device, chunk_size=64, augment=False):
        self.bb_files = parquets
        self.img_files = imgs
        self.augment = augment
        self.embedder = embedder
        self.device = device 
        self.chunk_size = chunk_size
        self.train_aug = A.Compose(
            [
                A.ToGray(num_output_channels=1, p=1.0),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Normalize(mean=[0.5], std=[0.5]),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.GaussNoise(p=0.2),
                A.GaussianBlur(blur_limit=3, p=0.1),
                ToTensorV2(),
            ]
        )
        self.val_aug = A.Compose(
            [
                A.ToGray(num_output_channels=1, p=1.0),
                A.Normalize(mean=[0.5], std=[0.5]),
                ToTensorV2(),
            ]
        )

        self.classes = list(map((lambda x: 0 if "EV" in x else 1), [f.name for f in self.bb_files]))
        self.bbs = {}
        for idx, file in enumerate(self.bb_files):
            self.bbs[file] = (extract_bbs(str(file), []),self.classes[idx])
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.bb_files)
        # return sum(len(f[0]) for f in self.bbs.values())

    def __getitem__(self, idx):        
        aug = self.train_aug if self.augment else self.val_aug

        fname_bb = self.bb_files[idx]
        img = self.img_files[idx]
        boxes, label = self.bbs[fname_bb]
        boxes = sorted(boxes, key=lambda b: b["y1"])
        vips_img = pyvips.Image.new_from_file(str(img), access="sequential")
        embeddings = []
        for i in range(0, len(boxes), self.chunk_size):
            chunk_boxes = boxes[i:i+self.chunk_size]
            for bb in chunk_boxes:
                
                cropped = vips_img.crop(bb["x1"],bb["y1"], bb["x2"]-bb["x1"], bb["y2"]-bb["y1"])
                patch = np.ndarray(
                    buffer=cropped.write_to_memory(),
                    dtype=np.uint8,
                    shape=[cropped.height, cropped.width, cropped.bands],
                )
                patch = patch[:,:,:3]
                result = aug(image=patch)
                img_tensor = result["image"].float().unsqueeze(0).to(self.device)
                del cropped, patch, result
                with torch.no_grad():
                    emb = self.embedder(img_tensor) 
                embeddings.append(emb.cpu())
                del img_tensor, emb
            print(f"chunk {i} done")
        del vips_img 
        embeddings_final = torch.stack(embeddings, dim=0)
        return embeddings_final, torch.tensor(label, dtype=torch.float32)


## Training Loop

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
from torch.utils.data import DataLoader
pyvips.cache_set_max_mem(500 * 1024 * 1024)
pyvips.cache_set_max(0)

IMAGE_DIR = "/media/sdog/SSD-500/microglia_data"
IMAGE_DIR = "../data/"
imgs = [f for f in Path(IMAGE_DIR).glob("./*.tiff")]
PARQUET_DIR = "../cell_localization/bboxes"
parquets = [f for f in Path(PARQUET_DIR).glob("./*.parquet")]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1
NUM_EPOCHS = 3
LR = 1e-4
skf = StratifiedKFold(n_splits=6,shuffle=True,random_state=30)
labels  =  list(map((lambda x: 0 if "EV" in x else 1), [f.name for f in imgs]))
embedder = MicrogliaFeatureEmbedder().to(DEVICE)


for fold, (train_idx, val_idx) in enumerate(skf.split(imgs, labels)):
    train_ids = [imgs[i] for i in train_idx]
    val_ids = [imgs[i] for i in val_idx]
    train_parquets = [parquets[i] for i in train_idx]
    val_parquets = [parquets[i] for i in val_idx]
    train_labels = [labels[i] for i in train_idx]
    val_labels = [labels[i] for i in val_idx]

    train_dataset = CellDataset(train_ids, train_parquets, embedder, DEVICE, augment=True)
    val_dataset = CellDataset(val_ids, val_parquets, embedder, DEVICE, augment=False)


    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = MILClassifier().to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS
    )

    best_val_loss = float("inf")
    best_state = None


        
    from tqdm import tqdm

    for epoch in tqdm(range(NUM_EPOCHS)):

        model.train()
        train_loss = 0.0
        for embeds, label in train_loader:

            embs = embeds.squeeze(0).to(DEVICE)
            label = label.squeeze(0).to(DEVICE)

            optimizer.zero_grad()
            pred, attn = model(embs)
            loss = loss_fn(pred.squeeze(), label.squeeze())

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        n = len(train_loader)
        train_loss = train_loss / n


        # TODO: VALIDATION STEP
        model.eval()
        val_loss = 0.0

        with torch.inference_mode():
            for embeds, label in val_loader:
                
                embs = embeds.squeeze(0).to(DEVICE)
                label = label.squeeze(0).to(DEVICE)
                pred, attn = model(embs)
                val_loss += loss_fn(pred.squeeze(), label.squeeze()).item()

        val_loss /= len(val_loader)
        scheduler.step()

        print(f"Epoch {epoch+1}/{NUM_EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                },
                "./checkpoints/best_model.pth",
            )
            print(f"Saved best model: {val_loss:.4f}")

        gc.collect()

    print("Training complete. Best val loss:", best_val_loss)

## Prediction

In [ ]:
from pathlib import Path
import torch 

IMAGE_DIR = "/media/sdog/SSD-500/microglia_data"
IMAGE_DIR = "../data"
imgs = [f for f in Path(IMAGE_DIR).glob("./*.tiff")]
PARQUET_DIR = "../cell_localization/bboxes"
parquets = [f for f in Path(PARQUET_DIR).glob("./*.parquet")]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1
LR = 1e-4

model = MILClassifier().to(DEVICE)

checkpoint = torch.load("./checkpoints/mil_attn.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

dataset = CellDataset(imgs, parquets)
embedder = MicrogliaFeatureEmbedder().to(DEVICE)
loader = DataLoader(dataset, BATCH_SIZE, num_workers=0)

all_records = []

with torch.inference_mode():
    attn_vector = []
    preds = []
    gt_labels = []
    for idx, (embeds, label) in enumerate(train_loader):

        embs = embeds.squeeze(0).to(DEVICE)
        label = label.squeeze(0).to(DEVICE)

        optimizer.zero_grad()
        pred, attn = model(embs)
        attn_vector.append(attn.numpy())

        prob = torch.sigmoid(pred).item()
        gt = int(label.item())
        preds.append(prob)

        bb_file = parquets[idx]
        bbs = extract_bbs(str(bb_file), [])
        assert len(bbs) == len(attn_vector), f"Num bbs does not match num attn weights"

    data = {
        "prediction": preds,
        "attn": attn_vector
    }
    df = pd.DataFrame(data, index=[f.name for f in imgs])
    df.to_json('attn.json', orient='split', compression='infer', index=True)
    

## Attention Map